In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/CMSC472Final')

!pip install -r /content/drive/MyDrive/CMSC472Final/requirements_colab.txt

In [2]:
from google.colab import auth
auth.authenticate_user()

In [3]:
from google.cloud import storage
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm import tqdm

BUCKET = "deeplearning-cmsc472-audio-dataset"
PREFIX = "rawstems/"
LOCAL_ROOT = Path("/content/rawstems")
WORKERS = 16

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
client = storage.Client()
bucket = client.bucket(BUCKET)

blobs = list(bucket.list_blobs(prefix=PREFIX))
print(f"Found {len(blobs)} blobs")

def download_blob(blob):
    rel = blob.name[len(PREFIX):]
    if not rel:
        return ("skip", "")
    dest = LOCAL_ROOT / rel
    if dest.exists() and dest.stat().st_size == blob.size:
        return ("skip", rel)
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        blob.download_to_filename(str(dest))
        return ("ok", rel)
    except Exception as e:
        return ("FAIL", f"{rel}: {e}")

results = {"ok": 0, "skip": 0, "FAIL": []}
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = [pool.submit(download_blob, b) for b in blobs]
    for fut in tqdm(as_completed(futures), total=len(futures)):
        status, info = fut.result()
        if status == "FAIL":
            results["FAIL"].append(info)
        else:
            results[status] += 1

print(f"\nok={results['ok']}, skipped={results['skip']}, failed={len(results['FAIL'])}")
for f in results["FAIL"][:20]:
    print(f"  {f}")


!gsutil cp gs://deeplearning-cmsc472-audio-dataset/manifest.csv /content/manifest.csv

Found 3079 blobs


100%|██████████| 3079/3079 [07:11<00:00,  7.14it/s]



ok=3079, skipped=0, failed=0
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://deeplearning-cmsc472-audio-dataset/manifest.csv...
/ [1 files][ 77.4 KiB/ 77.4 KiB]                                                
Operation completed over 1 objects/77.4 KiB.                                     


In [4]:
!gsutil cp gs://deeplearning-cmsc472-audio-dataset/manifest.csv /content/manifest.csv

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://deeplearning-cmsc472-audio-dataset/manifest.csv...
/ [1 files][ 77.4 KiB/ 77.4 KiB]                                                
Operation completed over 1 objects/77.4 KiB.                                     


In [ ]:
  import sys
  sys.path.append('/content/drive/MyDrive/CMSC472Final')
  from training import train, test, plot_metrics
  from model import StemSeparator
  import torch
  CONFIG_FILE = '/content/drive/MyDrive/CMSC472Final/weights/roformer-model-bs-roformer-sw-by-jarredou/BS-Rofo-SW-Fixed.yaml'
  CHECKPOINT_FILE  ='/content/drive/MyDrive/CMSC472Final/weights/roformer-model-bs-roformer-sw-by-jarredou/BS-Rofo-SW-Fixed.ckpt'

  SAVE_DIR       ='/content/drive/MyDrive/CMSC472Final/checkpoints'

  history = train(
      config_path      = CONFIG_FILE,
      checkpoint_path  = CHECKPOINT_FILE,
      save_dir         = SAVE_DIR,
      epochs           = 40,
      warmup_epochs    = 4,
      batch_size       = 8,
      lr_pretrained    = 2.5e-5,
      lr_new           = 2.5e-4,
      lr_warmup_new    = 2.5e-6,
      num_workers      = 2,
  )

  # -----------------------------------------------------------------------
  # Evaluate & plot
  # -----------------------------------------------------------------------
  device = 'cuda' if torch.cuda.is_available() else 'cpu'
  model  = StemSeparator(CONFIG_FILE, CHECKPOINT_FILE, device=device)
  model.load_state_dict(torch.load(
      f'{SAVE_DIR}/best_model.pt', map_location='cpu'
  ))



Gated fusion could be heavily downweighting the temporal block